# Chapter 2 — The Shock Hits

## Research question

How did the COVID-19 pandemic affect public health and economic growth across our selected countries between 2020 and 2022?

## Objectives

- Compare the timing and severity of the pandemic across countries.
-Examine trends in reported COVID-19 mortality and vaccination across countries during the pandemic.
- Compare annual GDP growth across countries during the pandemic.
- Explore whether countries with greater recorded health impacts also experienced weaker economic growth.
- Prepare evidence-based findings and visualizations for the project dashboard.

## Data sources

- **Our World in Data (OWID):** COVID-19 cases, deaths, vaccination, and excess mortality where available.
- **World Bank:** Annual GDP growth.
- **IMF October 2019 forecasts:** Pre-pandemic GDP growth expectations, for comparison once the forecast comparison method is aligned with Chapter 1.

## Important limitation

These comparisons are descriptive. Relationships between health outcomes and GDP growth do not, by themselves, establish that one caused the other. Country differences, missing data, and policy responses must be considered.

In [26]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Find the project folder reliably
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"

print("Project root:", ROOT)
print("Processed data folder exists:", DATA_DIR.exists())
print("\nAvailable processed CSV files:")
for file in sorted(DATA_DIR.glob("*.csv")):
    print("-", file.name)

Project root: /Users/yashviseksaria/theworldthatneverhappened
Processed data folder exists: True

Available processed CSV files:
- imf_gdp_growth_oct2019.csv
- imf_gdp_growth_oct2019_long.csv
- owid_covid_health_2020_2022.csv
- oxcgrt_policy_2020_2022.csv
- world_bank_gdp_growth.csv
- world_bank_vulnerability_2019.csv


In [27]:
health = pd.read_csv(DATA_DIR / "owid_covid_health_2020_2022.csv")
gdp = pd.read_csv(DATA_DIR / "world_bank_gdp_growth.csv")
imf = pd.read_csv(DATA_DIR / "imf_gdp_growth_oct2019_long.csv")

print("Health data:", health.shape)
print("World Bank GDP data:", gdp.shape)
print("IMF forecast data:", imf.shape)

display(health.head())
display(gdp.head())
display(imf.head())

Health data: (12056, 7)
World Bank GDP data: (143, 4)
IMF forecast data: (66, 7)


,country,code,date,new_cases_smoothed_per_million,new_deaths_smoothed_per_million,people_fully_vaccinated_per_hundred,excess_mortality
0,Brazil,BRA,2020-01-01,NaN,NaN,NaN,NaN
1,Brazil,BRA,2020-01-02,NaN,NaN,NaN,NaN
2,Brazil,BRA,2020-01-03,NaN,NaN,NaN,NaN
3,Brazil,BRA,2020-01-04,NaN,NaN,NaN,NaN
4,Brazil,BRA,2020-01-05,NaN,NaN,NaN,NaN


,ISO3,country,year,gdp_growth_pct
0,USA,United States,2010,2.695193
1,ESP,Spain,2010,0.094122
2,DEU,Germany,2010,4.134638
3,SWE,Sweden,2010,5.750765
4,CHN,China,2010,10.591690


,ISO3,Country,WEO Country Code,WEO Subject Code,Estimates Start After,year,imf_gdp_growth_pct
0,BRA,Brazil,223,NGDP_RPCH,2018,2019,0.875
1,CHN,China,924,NGDP_RPCH,2018,2019,6.139
2,DEU,Germany,134,NGDP_RPCH,2018,2019,0.544
3,IND,India,534,NGDP_RPCH,2018,2019,6.124
4,KOR,Korea,542,NGDP_RPCH,2018,2019,1.954


In [28]:

# 1. Convert dates and years to consistent types
health["date"] = pd.to_datetime(health["date"], errors="coerce")
gdp["year"] = pd.to_numeric(gdp["year"], errors="coerce").astype("Int64")
imf["year"] = pd.to_numeric(imf["year"], errors="coerce").astype("Int64")

# 2. Inspect health-data coverage by country
health_coverage = (
    health.groupby(["code", "country"], dropna=False)
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        rows=("date", "size"),
        unique_dates=("date", "nunique"),
        cases_available=("new_cases_smoothed_per_million", "count"),
        deaths_available=("new_deaths_smoothed_per_million", "count"),
        vaccination_available=("people_fully_vaccinated_per_hundred", "count"),
        excess_mortality_available=("excess_mortality", "count"),
    )
    .reset_index()
)

display(health_coverage)

# 3. Missing values across the three datasets
missingness = pd.DataFrame({
    "health_missing_pct": health.isna().mean() * 100,
    "gdp_missing_pct": gdp.isna().mean() * 100,
    "imf_missing_pct": imf.isna().mean() * 100,
})

display(missingness.round(2))

# 4. Duplicate checks
print("Duplicate health country-date rows:",
      health.duplicated(["code", "date"]).sum())

print("Duplicate GDP country-year rows:",
      gdp.duplicated(["ISO3", "year"]).sum())

print("Duplicate IMF country-year rows:",
      imf.duplicated(["ISO3", "year"]).sum())

# 5. Confirm the year ranges
print("\nHealth date range:",
      health["date"].min(), "to", health["date"].max())

print("GDP years:",
      sorted(gdp["year"].dropna().unique().tolist()))

print("IMF forecast years:",
      sorted(imf["year"].dropna().unique().tolist()))


,code,country,first_date,last_date,rows,unique_dates,cases_available,deaths_available,vaccination_available,excess_mortality_available
0,BRA,Brazil,2020-01-01,2022-12-31,1096,1096,1088,1088,635,36
1,CHN,China,2020-01-01,2022-12-31,1096,1096,1088,1088,56,0
2,DEU,Germany,2020-01-01,2022-12-31,1096,1096,1088,1088,735,156
3,ESP,Spain,2020-01-01,2022-12-31,1096,1096,1088,1088,328,156
4,IND,India,2020-01-01,2022-12-31,1096,1096,1088,1088,694,0
5,KOR,South Korea,2020-01-01,2022-12-31,1096,1096,1088,1088,655,156
6,SAU,Saudi Arabia,2020-01-01,2022-12-31,1096,1096,1088,1088,241,0
7,SWE,Sweden,2020-01-01,2022-12-31,1096,1096,1088,1088,103,156
8,THA,Thailand,2020-01-01,2022-12-31,1096,1096,1088,1088,535,36
9,USA,United States,2020-01-01,2022-12-31,1096,1096,1088,1088,749,156


,health_missing_pct,gdp_missing_pct,imf_missing_pct
Country,NaN,NaN,0.0
Estimates Start After,NaN,NaN,0.0
ISO3,NaN,0.0,0.0
WEO Country Code,NaN,NaN,0.0
WEO Subject Code,NaN,NaN,0.0
code,0.00,NaN,NaN
country,0.00,0.0,NaN
date,0.00,NaN,NaN
excess_mortality,91.64,NaN,NaN
gdp_growth_pct,NaN,0.0,NaN


Duplicate health country-date rows: 0
Duplicate GDP country-year rows: 0
Duplicate IMF country-year rows: 0

Health date range: 2020-01-01 00:00:00 to 2022-12-31 00:00:00
GDP years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
IMF forecast years: [2019, 2020, 2021, 2022, 2023, 2024]


## 1. GDP Growth During the Pandemic (2019–2022)

Compare GDP growth across countries before and during COVID-19 to identify the 2020 economic shock and subsequent recovery.

Negative growth indicates economic contraction; positive growth indicates expansion, but not necessarily a full recovery.

In [29]:

gdp_period = gdp[gdp["year"].between(2019, 2022)].copy()

fig = px.line(
    gdp_period,
    x="year",
    y="gdp_growth_pct",
    color="country",
    markers=True,
    title="Annual GDP Growth Across Countries, 2019–2022",
    labels={
        "year": "Year",
        "gdp_growth_pct": "Real GDP growth (%)",
        "country": "Country"
    }
)

fig.add_hline(y=0, line_dash="dash", line_color="gray")

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Country"
)

fig.show(renderer="browser")


## 2. The Human Shock: COVID-19 Mortality

Reported COVID-19 deaths provide one measure of the pandemic's human impact. Comparing average daily deaths per million people across countries from 2020 to 2022 helps reveal differences in the severity of the reported mortality burden. Annual averages provide a broad comparison, while the monthly analysis in the next section shows when mortality waves peaked and how their intensity changed over time.

Reported deaths do not capture the full health impact of COVID-19, and cross-country comparisons are affected by differences in testing, death registration, and reporting practices.

In [30]:

# Prepare daily health data
health["date"] = pd.to_datetime(health["date"])
health["year"] = health["date"].dt.year

# Average daily reported cases and deaths per million by country and year
health_yearly = (
    health[health["year"].between(2020, 2022)]
    .groupby(["country", "code", "year"], as_index=False)
    .agg(
        avg_daily_cases_per_million=(
            "new_cases_smoothed_per_million", "mean"
        ),
        avg_daily_deaths_per_million=(
            "new_deaths_smoothed_per_million", "mean"
        ),
        days_with_case_data=(
            "new_cases_smoothed_per_million", "count"
        ),
        days_with_death_data=(
            "new_deaths_smoothed_per_million", "count"
        ),
    )
)

display(health_yearly.head(10))


,country,code,year,avg_daily_cases_per_million,avg_daily_deaths_per_million,days_with_case_data,days_with_death_data
0,Brazil,BRA,2020,99.156322,2.534779,358,358
1,Brazil,BRA,2021,192.909491,5.571252,365,365
2,Brazil,BRA,2022,181.757951,0.971794,365,365
3,China,CHN,2020,0.188970,0.009369,358,358
4,China,CHN,2021,0.067111,0.001766,365,365
5,China,CHN,2022,143.748762,0.074730,365,365
6,Germany,DEU,2020,53.832482,1.507191,358,358
7,Germany,DEU,2021,174.831618,2.350350,365,365
8,Germany,DEU,2022,985.797700,1.571442,365,365
9,India,IND,2020,20.001484,0.289825,358,358


In [31]:

fig = px.line(
    health_yearly,
    x="year",
    y="avg_daily_deaths_per_million",
    color="country",
    markers=True,
    title="Average Daily COVID-19 Deaths per Million, 2020–2022",
    labels={
        "year": "Year",
        "avg_daily_deaths_per_million": "Average daily deaths per million",
        "country": "Country",
    },
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Country",
)

fig.show(renderer="browser")


## 3. When Did the Pandemic Hit Hardest?

Monthly trends reveal changes in reported COVID-19 deaths that annual averages can hide. Comparing countries helps show how the timing and intensity of pandemic waves differed.


In [32]:

# Aggregate daily observations into monthly averages
health["month"] = health["date"].dt.to_period("M").dt.to_timestamp()

health_monthly = (
    health[health["date"].between("2020-01-01", "2022-12-31")]
    .groupby(["country", "code", "month"], as_index=False)
    .agg(
        avg_daily_deaths_per_million=(
            "new_deaths_smoothed_per_million", "mean"
        )
    )
)

fig = px.line(
    health_monthly,
    x="month",
    y="avg_daily_deaths_per_million",
    color="country",
    title="Monthly Average Daily COVID-19 Deaths per Million, 2020–2022",
    labels={
        "month": "Month",
        "avg_daily_deaths_per_million": "Average daily deaths per million",
        "country": "Country",
    },
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Country",
)

fig.show(renderer="browser")


## 4. Vaccination During the Pandemic

Compare the share of each country's population fully vaccinated over time. Coverage and reporting vary, so missing observations should not be interpreted as zero vaccination.

In [33]:

vaccination = health.dropna(
    subset=["people_fully_vaccinated_per_hundred"]
).copy()

vaccination["month"] = vaccination["date"].dt.to_period("M").dt.to_timestamp()

vaccination_monthly = (
    vaccination.groupby(["country", "code", "month"], as_index=False)
    ["people_fully_vaccinated_per_hundred"].mean()
)

fig = px.line(
    vaccination_monthly,
    x="month",
    y="people_fully_vaccinated_per_hundred",
    color="country",
    title="Full Vaccination Coverage by Country, 2020–2022",
    labels={
        "month": "Month",
        "people_fully_vaccinated_per_hundred": "Population fully vaccinated (%)",
        "country": "Country",
    },
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Country",
)

fig.show(renderer="browser")


## 5. Health Impact and Economic Growth

Compare reported COVID-19 mortality with annual GDP growth to explore whether countries with higher reported mortality also experienced weaker economic performance. This is a descriptive comparison and cannot establish causation.


In [34]:

# Average daily COVID-19 deaths per million by country and year
mortality_yearly = (
    health_yearly[
        ["code", "country", "year", "avg_daily_deaths_per_million"]
    ]
    .copy()
)

# Keep GDP observations for the pandemic years
gdp_pandemic = gdp[gdp["year"].between(2020, 2022)].copy()

# Join on country code and year
health_economy = mortality_yearly.merge(
    gdp_pandemic[["ISO3", "year", "gdp_growth_pct"]],
    left_on=["code", "year"],
    right_on=["ISO3", "year"],
    how="inner",
    validate="one_to_one"
)

print("Matched country-year observations:", len(health_economy))
display(health_economy.head(10))


Matched country-year observations: 33


,code,country,year,avg_daily_deaths_per_million,ISO3,gdp_growth_pct
0,BRA,Brazil,2020,2.534779,BRA,-3.276759
1,BRA,Brazil,2021,5.571252,BRA,4.762604
2,BRA,Brazil,2022,0.971794,BRA,3.016694
3,CHN,China,2020,0.009369,CHN,2.339504
4,CHN,China,2021,0.001766,CHN,8.570384
5,CHN,China,2022,0.074730,CHN,3.133854
6,DEU,Germany,2020,1.507191,DEU,-4.131914
7,DEU,Germany,2021,2.350350,DEU,3.910000
8,DEU,Germany,2022,1.571442,DEU,1.809258
9,IND,India,2020,0.289825,IND,-5.777725


In [35]:

fig = px.scatter(
    health_economy,
    x="avg_daily_deaths_per_million",
    y="gdp_growth_pct",
    color="country",
    symbol="year",
    hover_data=["year"],
    title="COVID-19 Mortality and GDP Growth, 2020–2022",
    labels={
        "avg_daily_deaths_per_million": "Average daily deaths per million",
        "gdp_growth_pct": "Real GDP growth (%)",
        "country": "Country",
        "year": "Year",
    },
)

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_white")
fig.show(renderer="browser")


In [36]:

valid_pairs = health_economy[
    ["avg_daily_deaths_per_million", "gdp_growth_pct"]
].dropna()

print("Valid country-year observations:", len(valid_pairs))
print(
    "Correlation:",
    round(
        valid_pairs["avg_daily_deaths_per_million"].corr(
            valid_pairs["gdp_growth_pct"]
        ),
        3
    )
)


Valid country-year observations: 33
Correlation: -0.11


## 6. Health Impact vs GDP Forecast Error

This section compares reported COVID-19 mortality with the gap between actual GDP growth and the IMF's October 2019 pre-pandemic forecast.

Forecast error is calculated as actual GDP growth minus forecast GDP growth, in percentage points. Negative values indicate that actual growth fell below expectations; positive values indicate that it exceeded expectations.

The analysis explores whether countries with higher reported mortality also experienced larger economic shortfalls relative to pre-pandemic expectations. This comparison is descriptive and cannot establish that mortality caused the economic shortfall.

In [37]:
# Load the IMF October 2019 GDP forecasts
imf_forecast = pd.read_csv(
    DATA_DIR / "imf_gdp_growth_oct2019_long.csv"
)

# Inspect the forecast and actual GDP datasets
print("IMF forecast shape:", imf_forecast.shape)
print("IMF forecast columns:", imf_forecast.columns.tolist())

print("\nActual GDP columns:", gdp.columns.tolist())

display(imf_forecast.head(10))

IMF forecast shape: (66, 7)
IMF forecast columns: ['ISO3', 'Country', 'WEO Country Code', 'WEO Subject Code', 'Estimates Start After', 'year', 'imf_gdp_growth_pct']

Actual GDP columns: ['ISO3', 'country', 'year', 'gdp_growth_pct']


,ISO3,Country,WEO Country Code,WEO Subject Code,Estimates Start After,year,imf_gdp_growth_pct
0,BRA,Brazil,223,NGDP_RPCH,2018,2019,0.875
1,CHN,China,924,NGDP_RPCH,2018,2019,6.139
2,DEU,Germany,134,NGDP_RPCH,2018,2019,0.544
3,IND,India,534,NGDP_RPCH,2018,2019,6.124
4,KOR,Korea,542,NGDP_RPCH,2018,2019,1.954
5,SAU,Saudi Arabia,456,NGDP_RPCH,2018,2019,0.167
6,ZAF,South Africa,199,NGDP_RPCH,2018,2019,0.655
7,ESP,Spain,184,NGDP_RPCH,2018,2019,2.175
8,SWE,Sweden,144,NGDP_RPCH,2018,2019,0.936
9,THA,Thailand,578,NGDP_RPCH,2018,2019,2.871


In [38]:
# Keep the forecast period from 2019 to 2022
imf_period = imf_forecast[
    imf_forecast["year"].between(2019, 2022)
].copy()

actual_period = gdp[
    gdp["year"].between(2019, 2022)
].copy()

# Merge forecast and actual GDP growth by country and year
forecast_comparison = imf_period.merge(
    actual_period[["ISO3", "year", "gdp_growth_pct"]],
    on=["ISO3", "year"],
    how="inner",
    validate="one_to_one"
)

# Calculate forecast error in percentage points
forecast_comparison["forecast_error"] = (
    forecast_comparison["gdp_growth_pct"]
    - forecast_comparison["imf_gdp_growth_pct"]
)

print("Matched country-year observations:", len(forecast_comparison))
print("Countries represented:", forecast_comparison["ISO3"].nunique())
print("\nObservations by year:")
print(forecast_comparison.groupby("year").size())

display(
    forecast_comparison[
        [
            "ISO3",
            "Country",
            "year",
            "imf_gdp_growth_pct",
            "gdp_growth_pct",
            "forecast_error"
        ]
    ].sort_values(["year", "forecast_error"])
)

Matched country-year observations: 44
Countries represented: 11

Observations by year:
year
2019    11
2020    11
2021    11
2022    11
dtype: int64


,ISO3,Country,year,imf_gdp_growth_pct,gdp_growth_pct,forecast_error
3,IND,India,2019,6.124,3.871437,-2.252563
9,THA,Thailand,2019,2.871,2.114558,-0.756442
6,ZAF,South Africa,2019,0.655,0.259936,-0.395064
7,ESP,Spain,2019,2.175,1.961179,-0.213821
1,CHN,China,2019,6.139,6.067378,-0.071622
10,USA,United States,2019,2.350,2.583825,0.233825
0,BRA,Brazil,2019,0.875,1.220778,0.345778
4,KOR,Korea,2019,1.954,2.313802,0.359802
2,DEU,Germany,2019,0.544,0.977735,0.433735
5,SAU,Saudi Arabia,2019,0.167,1.651589,1.484589


### Visualising the Gap Between Expectations and Reality

The heatmap compares actual GDP growth with the IMF's October 2019 forecasts for each country from 2019 to 2022. Each cell represents the forecast error in percentage points.

Negative values indicate growth below the pre-pandemic forecast, while positive values indicate growth above it. Comparing years makes it possible to see the widespread shortfall in 2020 and whether countries subsequently returned to, or exceeded, their expected growth rates.

In [39]:
# Arrange forecast errors into a country-by-year matrix
error_matrix = forecast_comparison.pivot(
    index="Country",
    columns="year",
    values="forecast_error"
)

# Order countries by their average forecast error
error_matrix = error_matrix.loc[
    error_matrix.mean(axis=1).sort_values().index
]

fig = px.imshow(
    error_matrix,
    labels={
        "x": "Year",
        "y": "Country",
        "color": "Forecast Error (percentage points)"
    },
    x=error_matrix.columns,
    y=error_matrix.index,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    text_auto=".1f",
    title="The World That Never Happened: GDP Forecast Errors, 2019–2022"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    height=650,
    width=1050,
    margin=dict(l=140, r=50, t=100, b=70)
)

fig.update_xaxes(dtick=1)
fig.show(renderer="browser")

In [40]:
# Country with the largest negative forecast error in 2020
errors_2020 = forecast_comparison[
    forecast_comparison["year"] == 2020
]

print("Largest 2020 shortfall:")
display(
    errors_2020.nsmallest(3, "forecast_error")[
        ["Country", "imf_gdp_growth_pct",
         "gdp_growth_pct", "forecast_error"]
    ]
)

# Average forecast error by year
print("\nAverage forecast error by year:")
display(
    forecast_comparison.groupby("year")["forecast_error"]
    .agg(["mean", "median", "min", "max"])
    .round(2)
)

# Countries with the strongest positive deviations in 2021
print("\nLargest positive forecast errors in 2021:")
display(
    forecast_comparison[
        forecast_comparison["year"] == 2021
    ].nlargest(3, "forecast_error")[
        ["Country", "imf_gdp_growth_pct",
         "gdp_growth_pct", "forecast_error"]
    ]
)

Largest 2020 shortfall:


,Country,imf_gdp_growth_pct,gdp_growth_pct,forecast_error
14,India,7.033,-5.777725,-12.810725
18,Spain,1.847,-10.940071,-12.787071
20,Thailand,3.007,-6.093063,-9.100063



Average forecast error by year:


,mean,median,min,max
year,,,,
2019,0.08,0.23,-2.25,1.67
2020,-6.60,-5.38,-12.81,-2.92
2021,2.73,2.67,-1.93,4.98
2022,1.13,0.30,-2.57,9.60



Largest positive forecast errors in 2021:


,Country,imf_gdp_growth_pct,gdp_growth_pct,forecast_error
29,Spain,1.702,6.683144,4.981144
32,United States,1.746,6.152022,4.406022
27,Saudi Arabia,2.220,6.519597,4.299597


In [41]:
# Compare annual COVID-19 mortality with GDP forecast errors
mortality_forecast = health_yearly[
    ["code", "country", "year", "avg_daily_deaths_per_million"]
].merge(
    forecast_comparison[
        ["ISO3", "year", "imf_gdp_growth_pct",
         "gdp_growth_pct", "forecast_error"]
    ],
    left_on=["code", "year"],
    right_on=["ISO3", "year"],
    how="inner",
    validate="one_to_one"
)

# Focus on the pandemic years
mortality_forecast = mortality_forecast[
    mortality_forecast["year"].between(2020, 2022)
].dropna(
    subset=["avg_daily_deaths_per_million", "forecast_error"]
).copy()

print("Matched country-year observations:", len(mortality_forecast))
print("Countries represented:", mortality_forecast["country"].nunique())
print("\nCorrelation between mortality and GDP forecast error:")
print(
    round(
        mortality_forecast["avg_daily_deaths_per_million"].corr(
            mortality_forecast["forecast_error"]
        ),
        3
    )
)

print("\nCorrelation by year:")
display(
    mortality_forecast.groupby("year").apply(
        lambda group: group["avg_daily_deaths_per_million"].corr(
            group["forecast_error"]
        )
    ).rename("Correlation").to_frame().round(3)
)

display(mortality_forecast.head())

Matched country-year observations: 33
Countries represented: 11

Correlation between mortality and GDP forecast error:
0.073

Correlation by year:


/var/folders/n9/fc9t9s8x2y9ck5tmjctvnvdc0000gn/T/ipykernel_52679/3963109061.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mortality_forecast.groupby("year").apply(


,Correlation
year,
2020,-0.033
2021,0.230
2022,-0.102


,code,country,year,avg_daily_deaths_per_million,ISO3,imf_gdp_growth_pct,gdp_growth_pct,forecast_error
0,BRA,Brazil,2020,2.534779,BRA,2.039,-3.276759,-5.315759
1,BRA,Brazil,2021,5.571252,BRA,2.371,4.762604,2.391604
2,BRA,Brazil,2022,0.971794,BRA,2.358,3.016694,0.658694
3,CHN,China,2020,0.009369,CHN,5.819,2.339504,-3.479496
4,CHN,China,2021,0.001766,CHN,5.900,8.570384,2.670384


In [42]:
fig = px.scatter(
    mortality_forecast,
    x="avg_daily_deaths_per_million",
    y="forecast_error",
    color="year",
    hover_name="country",
    hover_data={
        "avg_daily_deaths_per_million": ":.2f",
        "forecast_error": ":.2f",
        "imf_gdp_growth_pct": ":.2f",
        "gdp_growth_pct": ":.2f",
        "year": True
    },
    color_continuous_scale="Viridis",
    title="Health Impact and Economic Shortfall, 2020–2022",
    labels={
        "avg_daily_deaths_per_million": "Average Daily COVID-19 Deaths per Million",
        "forecast_error": "GDP Forecast Error (percentage points)",
        "year": "Year"
    }
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    annotation_text="Actual growth matched forecast"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    width=1050,
    height=650
)

fig.show(renderer="browser")

## 7. Key Findings and Limitations

### Key Findings

- **Economic shock:** GDP growth fell sharply below pre-pandemic forecasts in 2020. Across 11 countries, the average GDP forecast error was **−6.60 percentage points**.
- **Uneven recovery:** The average forecast error shifted to **+2.73 percentage points in 2021**, with 10 of 11 countries recording growth above their IMF October 2019 forecasts.
- **Uneven mortality:** Monthly COVID-19 death rates peaked at different times and levels across countries, showing that the timing and intensity of the reported health shock varied.
- **Uneven vaccination:** Full-vaccination coverage increased at different speeds, with substantial differences between countries.
- **Health and economic growth:** Across 33 country-year observations from 2020–2022, the correlation between average daily COVID-19 deaths per million and annual GDP growth was **−0.11**, indicating a very weak negative linear relationship in the sample.
- **Health impact and economic shortfall:** The correlation between average daily COVID-19 deaths per million and GDP forecast error was **+0.073** across 33 country-year observations. Year-specific correlations were −0.033 in 2020, +0.230 in 2021, and −0.102 in 2022. These weak relationships suggest that reported mortality alone did not consistently explain differences in economic outcomes relative to pre-pandemic forecasts.
- **Different recovery paths:** The forecast-error heatmap shows that outcomes varied across countries. Spain recorded a positive forecast error of approximately **+4.98 percentage points in 2021**, while China remained approximately **−2.57 percentage points below forecast in 2022**. Saudi Arabia recorded a particularly large positive forecast error of **+9.60 percentage points in 2022**.

### Overall Interpretation

The COVID-19 pandemic created a major health and economic shock, but its effects differed across countries and over time. The widespread negative GDP forecast errors in 2020 show how sharply actual growth diverged from pre-pandemic expectations. The positive average forecast error in 2021 indicates that growth exceeded the October 2019 forecasts in most of the countries studied, although this does not necessarily mean that all pandemic-related economic losses had been recovered.

The health indicators add another dimension to this story. Mortality waves and vaccination progress varied across countries, but the weak correlations between mortality and economic outcomes suggest that reported deaths alone cannot explain differences in economic performance. Economic structure, disruptions to production and trade, changes in behaviour, and government responses may also have contributed. Chapter 3 examines government policy choices as part of this wider context.

Taken together, the findings show a widespread economic shock, uneven health impacts, and different recovery paths. These comparisons identify patterns in the data but do not establish that mortality or any single policy caused a particular economic outcome.

### Limitations

- Reported cases and deaths may differ from actual infections and mortality because of differences in testing, death registration, and reporting.
- Missing observations are not equivalent to zero.
- Annual GDP growth measures changes in output, while forecast errors measure the difference between actual growth and pre-pandemic expectations. Neither directly measures cumulative output lost.
- Forecast errors reflect unexpected developments beyond the pandemic, including other economic shocks and changes in economic conditions.
- Actual GDP estimates may have been revised after the IMF published its October 2019 forecasts, so forecast errors may partly reflect later data revisions.
- Correlations between mortality, GDP growth, and forecast errors are descriptive and do not establish causation.
- The analysis covers 11 selected countries, so findings should not automatically be generalized to all countries.
- Vaccination coverage and mortality reporting vary across countries, so comparisons should be interpreted cautiously.

## 8. Dashboard Handoff

### Recommended Visuals

- **GDP growth:** Show the economic contraction in 2020 and subsequent growth across the selected countries from 2019 to 2022.
- **GDP forecast errors:** Include the country-by-year heatmap comparing actual GDP growth with the IMF's October 2019 forecasts. Highlight the widespread negative forecast errors in 2020 and the broad rebound above forecasts in 2021, while showing differences across countries.
- **Monthly COVID-19 deaths:** Compare the timing and intensity of reported mortality waves across countries from 2020 to 2022.
- **Vaccination coverage:** Show differences in the pace and extent of COVID-19 vaccination rollout across countries.
- **Health and economic growth:** Include the scatter plot comparing average daily deaths per million with annual GDP growth. Label the correlation of **−0.11** as a very weak negative descriptive relationship, not evidence of causation.
- **Health impact and economic shortfall:** Include the scatter plot comparing average daily deaths per million with GDP forecast error. Report the correlation of **+0.073** and explain that the relationship was very weak across the 33 country-year observations. Emphasize that mortality alone did not consistently explain economic shortfalls relative to pre-pandemic forecasts.

### Visualization Guidelines

- Use a consistent, colourblind-friendly country palette where country colours are used.
- Ensure countries remain distinguishable, especially when several appear on the same chart.
- Label axes, units, years, and data sources clearly.
- For the forecast-error heatmap, use a diverging colour scale centred at zero so negative and positive errors are easy to distinguish.
- Distinguish annual GDP growth rates from forecast errors and cumulative economic losses; these measure different things.
- Display missing data as unavailable rather than treating missing values as zero.
- Use consistent country names and colours across visualizations to make comparisons easier.

### Interpretation and Limitations

- Avoid claiming that COVID-19 mortality or government policies directly caused differences in GDP growth. The analysis is descriptive and does not establish causation.
- Explain that forecast errors measure deviations from pre-pandemic expectations and reflect unexpected developments, not the isolated economic impact of COVID-19.
- Explain that reported COVID-19 cases, deaths, and vaccination figures may differ from actual outcomes because data collection and reporting vary across countries.
- Make the main story clear: the pandemic produced a major economic shock, health impacts differed across countries and over time, and economic outcomes relative to pre-pandemic forecasts varied during the recovery.
- Clarify that growth above forecast does not necessarily mean that all economic losses were recovered.
- Keep charts readable and focused, with informative titles and concise annotations highlighting important patterns without overstating the findings.